In [16]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
%pip install -q -U \
    transformers \
    peft \
    trl \
    bitsandbytes \
    accelerate \
    datasets \
    pyyaml \
    sentencepiece

In [18]:
from google.colab import drive
from pathlib import Path
import zipfile
import shutil
import sys

drive.mount("/content/drive")

DRIVE_DIR = Path(
    "/content/drive/MyDrive/LLMProjects/fine-tune"
)

assert DRIVE_DIR.exists(), f"Папка {DRIVE_DIR} не существует"

zip_files = sorted(DRIVE_DIR.glob('*.zip'))

print("\nZip files")
for path in zip_files:
  print(path.name)

assert len(zip_files) == 1, (
    "Ожидался один файл zip в папке.\n",
    f"Найдено: {[path.name for path in zip_files]}"
)

ARCHIVE_PATH = zip_files[0]

WORKDIR = Path("/content/qlora-routing-support-tickets")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

WORKDIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ARCHIVE_PATH, "r") as zip_file:
    zip_file.extractall(WORKDIR)

candidate_roots = [WORKDIR] + [
    path for path in WORKDIR.iterdir() if path.is_dir()
]

PROJECT_ROOT = next(
    (
        path
        for path in candidate_roots
        if (path / "src/").is_dir() and (path / "data").is_dir()
    ),
    None,
)

assert PROJECT_ROOT is not None, (
    "После распаковки не найдена папка проекта с data/ и src/.\n"
    f"Содержимое {WORKDIR}: {[path.name for path in WORKDIR.iterdir()]}"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

OUTPUTS_DIR = DRIVE_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("\nPATHS")
print("Archive:     ", ARCHIVE_PATH)
print("Project root:", PROJECT_ROOT)
print("Outputs:     ", OUTPUTS_DIR)

required_files = [
    "data/processed/train.jsonl",
    "data/processed/validation.jsonl",
    "data/processed/test.jsonl",
    "src/schemas.py",
    "src/prediction_utils.py",
]

print("\nRequired files")
missing_files = []

for relative_path in required_files:
    path = PROJECT_ROOT / relative_path
    exists = path.exists()
    print(f"{relative_path}: {exists}")

    if not exists:
        missing_files.append(relative_path)

assert not missing_files, (
    "\nНе найдены обязательные файлы" + "\n".join(missing_files)
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Zip files
qlora-routing-support-tickets.zip

PATHS
Archive:      /content/drive/MyDrive/LLMProjects/fine-tune/qlora-routing-support-tickets.zip
Project root: /content/qlora-routing-support-tickets
Outputs:      /content/drive/MyDrive/LLMProjects/fine-tune/outputs

Required files
data/processed/train.jsonl: True
data/processed/validation.jsonl: True
data/processed/test.jsonl: True
src/schemas.py: True
src/prediction_utils.py: True


# Train Qwen-QLoRA

In [4]:
import gc
import json
import numpy as np

from datasets import DatasetDict, load_dataset
from transformers import AutoTokenizer

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

In [5]:
torch.cuda.is_available()

True

In [6]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "data" / "processed" / "validation.jsonl"

assert TRAIN_PATH.exists(), f"Не найден train: {TRAIN_PATH}"
assert VALIDATION_PATH.exists(), f"Не найден validation: {VALIDATION_PATH}"


raw_datasets = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_PATH),
        "validation": str(VALIDATION_PATH),
    },
)

# raw_datasets = DatasetDict(
#     {
#         "train": load_dataset(
#             "json",
#             data_files=str(TRAIN_PATH),
#             split="train",
#         ),
#         "validation": load_dataset(
#             "json",
#             data_files=str(VALIDATION_PATH),
#             split="train",
#         )
#     }
# )

print("Raw datasets")
print(raw_datasets)

print("Исходные данные")
print(raw_datasets["train"].column_names)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Raw datasets
DatasetDict({
    train: Dataset({
        features: ['messages', 'target'],
        num_rows: 1200
    })
    validation: Dataset({
        features: ['messages', 'target'],
        num_rows: 150
    })
})
Исходные данные
['messages', 'target']


In [7]:
def convert_to_prompt_completion(example: dict) -> dict:
    messages = example["messages"]

    assert len(messages) == 3, (
        f"Ожидалось 3 сообщения, получено {len(messages)}"
    )

    roles = [message["role"] for message in messages]

    assert roles == ["system", "user", "assistant"], (
        f"Некорректный порядок ролей: {roles}"
    )

    return {
        "prompt": messages[:2],
        "completion": [messages[2]]
    }

prompt_completion_datasets = raw_datasets.map(
    convert_to_prompt_completion,
    remove_columns=raw_datasets["train"].column_names,
)

print("Prompt-completion datasets")
print(prompt_completion_datasets)

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Prompt-completion datasets
DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1200
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 150
    })
})


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("\nTokenizer")
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Pad token:", repr(tokenizer.pad_token))
print("Pad token id:", tokenizer.pad_token_id)
print("EOS token:", repr(tokenizer.eos_token))
print("EOS token id:", tokenizer.eos_token_id)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]


Tokenizer
Tokenizer class: Qwen2Tokenizer
Pad token: '<|endoftext|>'
Pad token id: 151643
EOS token: '<|im_end|>'
EOS token id: 151645


In [9]:
example = prompt_completion_datasets["train"][0]

formatted_prompt = tokenizer.apply_chat_template(
    example["prompt"],
    tokenize=False,
    add_generation_prompt=True,
)

formatted_full_example = tokenizer.apply_chat_template(
    example["prompt"] + example["completion"],
    tokenize=False,
    add_generation_prompt=False,
)

print("Formatted prompt")
print(formatted_prompt)
print("=" * 100)
print("Full training example")
print(formatted_full_example)

Formatted prompt
<|im_start|>system
You classify customer-support tickets into a fixed routing schema. Return exactly one valid JSON object and nothing else. Do not use Markdown code fences, explanations, comments, or extra keys.

Required keys: ticket_type, topic, urgency, tags.

Allowed ticket_type values: Incident, Request, Problem, Change.
Allowed topic values: Billing and Payments, Customer Service, General Inquiry, Human Resources, IT Support, Product Support, Returns and Exchanges, Sales and Pre-Sales, Service Outages and Maintenance, Technical Support.
Allowed urgency values: low, medium, high.

Use ticket_type, topic, and urgency exactly as listed, including capitalization and spaces. Do not invent new values.
tags must be a JSON array with up to 8 short, relevant, non-duplicate labels.<|im_end|>
<|im_start|>user
Subject: Incorporating Live Data Analytics

Body: Dear Customer Support Team,

I am seeking assistance with incorporating live data analytics based on Python machine 

In [10]:
def add_token_lengths(example: dict) -> dict:
    prompt_token = tokenizer.apply_chat_template(
        example["prompt"],
        tokenize=True,
        add_generation_prompt=True,
    )

    full_token = tokenizer.apply_chat_template(
        example["prompt"] + example["completion"],
        tokenize=True,
        add_generation_prompt=False,
    )

    return {
        "prompt_tokens": len(prompt_token['input_ids']),
        "full_tokens": len(full_token['input_ids']),
        "completion_tokens": len(full_token['input_ids']) - len(prompt_token['input_ids'])
    }

datasets_with_lengths = prompt_completion_datasets.map(add_token_lengths)

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [11]:
def print_length_stats(split_name: str):
    split = datasets_with_lengths[split_name]

    prompt_lengths = np.array(split["prompt_tokens"])
    full_lengths = np.array(split["full_tokens"])
    completion_lengths = np.array(split["completion_tokens"])

    print(f"{split_name.upper()} token lengths")

    for name, lengths in [
        ("Prompt", prompt_lengths),
        ("Completion", completion_lengths),
        ("Full example", full_lengths),
    ]:
        print(
            f"{name:14} | "
            f"min={lengths.min():4d} | "
            f"median={np.median(lengths):6.1f} | "
            f"p90={np.percentile(lengths, 90):6.1f} | "
            f"p95={np.percentile(lengths, 95):6.1f} | "
            f"p99={np.percentile(lengths, 99):6.1f} | "
            f"max={lengths.max():4d}"
        )

print_length_stats("train")
print()
print_length_stats("validation")

TRAIN token lengths
Prompt         | min= 182 | median= 268.0 | p90= 285.0 | p95= 289.0 | p99= 301.0 | max= 379
Completion     | min=  32 | median=  43.0 | p90=  51.0 | p95=  53.0 | p99=  55.0 | max=  59
Full example   | min= 218 | median= 312.0 | p90= 330.0 | p95= 333.0 | p99= 346.0 | max= 421

VALIDATION token lengths
Prompt         | min= 186 | median= 267.0 | p90= 284.1 | p95= 290.5 | p99= 307.0 | max= 361
Completion     | min=  33 | median=  43.0 | p90=  50.1 | p95=  52.0 | p99=  55.0 | max=  56
Full example   | min= 222 | median= 311.0 | p90= 328.1 | p95= 334.5 | p99= 347.5 | max= 411


In [12]:
train_with_lengths = datasets_with_lengths["train"]

longest_indices = sorted(
    range(len(train_with_lengths)),
    key=lambda index: train_with_lengths[index]['full_tokens'],
    reverse=True,
)[:5]

print("5 longest train examples")

for rank, index in enumerate(longest_indices, start=1):
    row = train_with_lengths[index]

    user_text = row["prompt"][1]["content"]
    subject_line = user_text.split("\n", maxsplit=1)[0]

    print(
        f"{rank}. index={index} | "
        f"full_tokens={row['full_tokens']} | "
        f"completion_tokens={row['completion_tokens']} | "
        f"{subject_line[:120]}"
    )

5 longest train examples
1. index=553 | full_tokens=421 | completion_tokens=42 | Subject: Request for Product Swap
2. index=1000 | full_tokens=411 | completion_tokens=44 | Subject: Urgent Update Needed for SaaS Platform
3. index=766 | full_tokens=398 | completion_tokens=38 | Subject: Request for Complete Product Support Documentation
4. index=26 | full_tokens=388 | completion_tokens=47 | Subject: Severe Service Disruption Detected in EMR Platform
5. index=244 | full_tokens=371 | completion_tokens=43 | Subject: Request for System Enhancement and Upgrade Integration


In [13]:
MAX_LENGTH = 512

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

assert max(datasets_with_lengths["train"]["full_tokens"]) <= MAX_LENGTH
assert max(datasets_with_lengths["validation"]["full_tokens"]) <= MAX_LENGTH

COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

print("qlora config")
print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)
print("Compute dtype:", COMPUTE_DTYPE)
print("LoRA rank:", LORA_R)
print("LoRA alpha:", LORA_ALPHA)
print("LoRA dropout:", LORA_DROPOUT)

qlora config
Model: Qwen/Qwen2.5-0.5B-Instruct
Max length: 512
Compute dtype: torch.bfloat16
LoRA rank: 8
LoRA alpha: 16
LoRA dropout: 0.05


In [14]:
gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    dtype=COMPUTE_DTYPE
)

base_model.config.use_cache = False

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [15]:
print("Model class: ", base_model.__class__.__name__)
print("4-bit loader: ", base_model.is_loaded_in_4bit)

base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=False)

Model class:  Qwen2ForCausalLM
4-bit loader:  True


In [16]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules="all-linear",
)

model = get_peft_model(base_model, lora_config)

print("Trainable parameters")
model.print_trainable_parameters()

trainable_names = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

print("First trainable parameter names:")
for name in trainable_names[:10]:
    print("-", name)

allocated_gb = torch.cuda.memory_allocated() / 1024**3
reserved_gb = torch.cuda.memory_reserved() / 1024**3

print("\nGPU memory after load")
print(f"Allocated: {allocated_gb:.2f} GB")
print(f"Reserved:  {reserved_gb:.2f} GB")

Trainable parameters
trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826
First trainable parameter names:
- base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
- base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
- base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight
- base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight
- base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
- base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
- base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight
- base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight
- base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight
- base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight

GPU memory after load
Allocated: 0.70 GB
Reserved:  0.97 GB


In [17]:
from torch.utils.data import Dataset, DataLoader


class TicketRoutingDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length: int):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, index: int) -> dict:
        example = self.dataset[index]

        prompt_tokenize = self.tokenizer.apply_chat_template(
            example["prompt"],
            tokenize=True,
            add_generation_prompt=True,
        )

        full_tokenize = self.tokenizer.apply_chat_template(
            example["prompt"] + example["completion"],
            tokenize=True,
            add_generation_prompt=False,
        )

        full_attention_mask = full_tokenize["attention_mask"]

        prompt_ids = prompt_tokenize["input_ids"]
        full_ids = full_tokenize["input_ids"]

        assert full_ids[:len(prompt_ids)] == prompt_ids
        assert len(full_ids) <= self.max_length

        input_ids = full_ids.copy()
        labels = full_ids.copy()

        labels[:len(prompt_ids)] = [-100] * len(prompt_ids)

        assert any(label != -100 for label in labels)

        return {
            "input_ids": input_ids,
            "attention_mask": full_attention_mask,
            "labels": labels,
        }


train_dataset = TicketRoutingDataset(
    prompt_completion_datasets["train"],
    tokenizer,
    MAX_LENGTH,
)

validation_dataset = TicketRoutingDataset(
    prompt_completion_datasets["validation"],
    tokenizer,
    MAX_LENGTH,
)

In [18]:
sample = train_dataset[0]

print("Длина input_ids: ", len(sample["input_ids"]))
print("Длина labels: ", len(sample["labels"]))
print("Реальных токенов: ", sum(sample["attention_mask"]))
print("Токенов с loss: ", sum(label != -100 for label in sample["labels"]))

completion_ids = [
    label
    for label in sample["labels"]
    if label != -100
]

print("Target for loss")
print(
    tokenizer.decode(
        completion_ids,
        skip_special_tokens=False,
    )
)

Длина input_ids:  309
Длина labels:  309
Реальных токенов:  309
Токенов с loss:  39
Target for loss
{"ticket_type": "Request", "topic": "Technical Support", "urgency": "high", "tags": ["Product", "Feature", "Documentation", "Tech Support"]}<|im_end|>



In [19]:
def collate_ticket_routing(batch: list[dict]) -> dict[str, torch.Tensor]:
    max_batch_length = max(len(example["input_ids"]) for example in batch)

    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []

    for example in batch:
        current_length = len(example["input_ids"])
        padding_length = max_batch_length - current_length

        input_ids = example["input_ids"] + [
            tokenizer.pad_token_id
        ] * padding_length

        attention_mask = example["attention_mask"] + [
            0
        ] * padding_length

        labels = example["labels"] + [
            -100
        ] * padding_length

        batch_input_ids.append(input_ids)
        batch_attention_mask.append(attention_mask)
        batch_labels.append(labels)

    result = {
        "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long),
        "labels": torch.tensor(batch_labels, dtype=torch.long),
    }

    assert result["input_ids"].shape == result["attention_mask"].shape
    assert result["input_ids"].shape == result["labels"].shape
    assert torch.all(result["labels"][result["attention_mask"] == 0] == -100)

    return result

In [20]:
BATCH_SIZE = 4
SEED = 42

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_ticket_routing,
    shuffle=True,
    generator=generator,
    pin_memory=True,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_ticket_routing,
    shuffle=False,
    pin_memory=True,
)

In [21]:
batch = next(iter(train_loader))

for name, tensor in batch.items():
    print(f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")

print("Real tokens per example:")
print(batch['attention_mask'].sum(dim=1))

print("Supervised tokens per example:")
print((batch['labels'] != -100).sum(dim=1))

print("\nPadding labels are all -100:")
print(bool(torch.all(batch["labels"][batch["attention_mask"] == 0] == -100)))

input_ids: shape=(4, 305), dtype=torch.int64
attention_mask: shape=(4, 305), dtype=torch.int64
labels: shape=(4, 305), dtype=torch.int64
Real tokens per example:
tensor([253, 305, 232, 265])
Supervised tokens per example:
tensor([36, 54, 39, 44])

Padding labels are all -100:
True


In [22]:
from torch.amp import GradScaler
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
import torch.nn as nn
import math
from peft import (
    get_peft_model_state_dict,
    set_peft_model_state_dict
)

class ManualQLoRATrainer:
    def __init__(self, model, optimizer, device, train_loader, validation_loader, gradient_accumulation_steps: int = 1, max_grad_norm: float | None = 1.0, scheduler=None):
        self.model = model
        self.optimizer = optimizer
        self.device = device

        self.scheduler = scheduler
        self.max_grad_norm = max_grad_norm
        self.gradient_accumulation_steps = gradient_accumulation_steps
        assert self.gradient_accumulation_steps >= 1

        self.train_loader = train_loader
        self.validation_loader = validation_loader

        self.global_step = 0

        self.use_amp = (self.device == "cuda")

        if self.use_amp and torch.cuda.is_bf16_supported():
            self.dtype = torch.bfloat16
        else:
            self.dtype = torch.float16

        self.scaler = GradScaler(
            enabled=(self.use_amp and self.dtype == torch.float16)
        )

        self.trainable_parameters = [
            parameter
            for parameter in self.model.parameters()
            if parameter.requires_grad
        ]

    def save_adapters(self, output_dir: str) -> None:
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)

        self.model.save_pretrained(output_path)

        print(f"LoRA adapter сохранён в: {output_path}")


    def _get_adapter_state(self) -> dict[str, torch.Tensor]:
        adapter_state = get_peft_model_state_dict(self.model)

        return {
            name: tensor.detach().cpu().clone()
            for name, tensor in adapter_state.items()
        }

    def _restore_adapter_state(
        self,
        adapter_state: dict[str, torch.Tensor],
    ) -> None:
        incompatible_keys = set_peft_model_state_dict(
            self.model,
            adapter_state,
        )

        assert not incompatible_keys.unexpected_keys, (
            f"Неожиданные ключи при восстановлении adapter: "
            f"{incompatible_keys.unexpected_keys}"
        )

        restored_state = get_peft_model_state_dict(self.model)

        assert set(restored_state) == set(adapter_state), (
            "После восстановления набор LoRA-весов не совпал "
            "с сохранённым состоянием."
        )

        for name, saved_tensor in adapter_state.items():
            restored_tensor = restored_state[name].detach().cpu()

            assert torch.equal(restored_tensor, saved_tensor), (
                f"LoRA-вес не совпал после восстановления: {name}"
            )

    def _move_batch_to_device(self, batch: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
        return {
            name: tensor.to(self.device, non_blocking=True)
            for name, tensor in batch.items()
        }

    def _compute_loss(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        with torch.autocast(
            device_type=self.device,
            dtype=self.dtype,
            enabled=self.use_amp
        ):
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

        return loss

    def train_epoch(self) -> dict[str, float]:
        self.model.train()

        self.optimizer.zero_grad(set_to_none=True)

        total_negative_log_likelihood = 0.0
        total_supervised_tokens = 0

        optimizer_steps_this_epoch = 0
        total_gradient_norm = 0.0

        pending_batches  = []

        train_progress = tqdm(
            enumerate(self.train_loader, start = 1),
            total=len(self.train_loader),
            desc="Train",
            unit="microbatch",
            leave=True,
        )

        for batch_index, cpu_batch in train_progress:
            pending_batches.append(cpu_batch)

            should_step = (
                len(pending_batches) == self.gradient_accumulation_steps
                or batch_index == len(self.train_loader)
            )

            if should_step:
                token_counts = [
                    (microbatch["labels"] != -100).sum().item()
                    for microbatch in pending_batches
                ]

                window_supervised_tokens = sum(token_counts)
                assert window_supervised_tokens > 0

                for cpu_microbatch, microbatch_tokens in zip(
                    pending_batches,
                    token_counts,
                ):
                    batch = self._move_batch_to_device(cpu_microbatch)

                    loss = self._compute_loss(**batch)

                    assert torch.isfinite(loss), (
                        f"Loss стал NaN или Inf: {loss.item()}"
                    )

                    total_negative_log_likelihood += loss.detach().float().item() * microbatch_tokens

                    total_supervised_tokens += microbatch_tokens

                    token_weight = microbatch_tokens / window_supervised_tokens

                    scaled_loss = loss * token_weight

                    self.scaler.scale(scaled_loss).backward()

                self.scaler.unscale_(self.optimizer)

                if self.max_grad_norm is not None:
                    gradient_norm = torch.nn.utils.clip_grad_norm_(
                        self.trainable_parameters,
                        max_norm=self.max_grad_norm,
                    )

                    assert torch.isfinite(gradient_norm), (
                        f"Gradient norm стал NaN или Inf: {gradient_norm.item()}"
                    )

                    total_gradient_norm += gradient_norm.detach().float().item()

                self.scaler.step(self.optimizer)
                self.scaler.update()

                optimizer_steps_this_epoch += 1
                self.global_step += 1

                if self.scheduler is not None:
                    self.scheduler.step()

                self.optimizer.zero_grad(set_to_none=True)

                running_train_loss = total_negative_log_likelihood / total_supervised_tokens

                postfix = {
                    "loss": f"{running_train_loss:.4f}",
                    "lr": f"{self.optimizer.param_groups[0]['lr']:.2e}",
                    "step": self.global_step,
                }

                if self.max_grad_norm is not None:
                    postfix["grad_norm"] = f"{gradient_norm.detach().float().item():.2f}"

                train_progress.set_postfix(postfix)

                pending_batches.clear()

        return {
            "train_loss": (
                total_negative_log_likelihood / total_supervised_tokens
            ),
            "mean_gradient_norm": total_gradient_norm / optimizer_steps_this_epoch,
            "learning_rate": self.optimizer.param_groups[0]["lr"],
            "optimizer_steps": optimizer_steps_this_epoch,
            "global_step": self.global_step,
        }

    @torch.inference_mode()
    def evaluate(self) -> dict[str, float]:
        self.model.eval()

        total_negative_log_likelihood = 0.0
        total_supervised_tokens = 0

        validation_progress = tqdm(
            self.validation_loader,
            total=len(self.validation_loader),
            desc="Validation",
            unit="batch",
            leave=True,
        )

        for batch in validation_progress:
            batch = self._move_batch_to_device(batch)

            loss = self._compute_loss(**batch)

            assert torch.isfinite(loss), (
                f"Loss стал NaN или Inf: {loss.item()}"
            )

            supervised_tokens = (batch["labels"] != -100).sum().item()

            total_negative_log_likelihood += (
                loss.detach().float().item()
                * supervised_tokens
            )

            total_supervised_tokens += supervised_tokens

            running_validation_loss = total_negative_log_likelihood / total_supervised_tokens

            validation_progress.set_postfix(
                loss=f"{running_validation_loss:.4f}"
            )

        return {
            "validation_loss": (
                total_negative_log_likelihood / total_supervised_tokens
            )
        }

    def fit(self, num_epochs: int, output_dir: str, early_stopping_patience: int | None = None, early_stopping_min_delta: float = 0.0) -> list[dict[str, float]]:
        history = []
        best_validation_loss = float("inf")
        bad_epochs = 0
        best_adapter_state = None

        for epoch in range(1, num_epochs + 1):
            train_metrics = self.train_epoch()
            validation_metrics = self.evaluate()

            metrics = {
                "epoch": epoch,
                **train_metrics,
                **validation_metrics,
            }

            history.append(metrics)

            print(
                f"Epoch {epoch}/{num_epochs} | "
                f"train_loss={metrics['train_loss']:.4f} | "
                f"validation_loss={metrics['validation_loss']:.4f} | "
                f"grad_norm={metrics['mean_gradient_norm']:.4f} | "
                f"lr={metrics['learning_rate']:.2e}"
            )

            improved = (
                metrics["validation_loss"]
                < best_validation_loss - early_stopping_min_delta
            )

            if improved:
                best_validation_loss = metrics["validation_loss"]
                best_adapter_state = self._get_adapter_state()
                bad_epochs = 0

                self.save_adapters(
                    f"{output_dir}/best_adapter"
                )
            else:
                bad_epochs += 1

                print(
                    f"Validation loss не улучшился. "
                    f"Bad epochs: {bad_epochs}"
                )

            if (
                early_stopping_patience is not None
                and bad_epochs >= early_stopping_patience
            ):
                print(
                    f"Early stopping: validation loss не улучшался "
                    f"{early_stopping_patience} эпохи подряд."
                )
                break

        assert best_adapter_state is not None

        self._restore_adapter_state(best_adapter_state)
        self.model.eval()

        print("В память восстановлен adapter с лучшим validation loss.")

        return history

In [23]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

learning_rate = 1e-4
weight_decay = 1e-3
gradient_accumulation_steps = 2
max_epochs = 8
warmup_ratio = 0.05
early_stopping_patience = 2
early_stopping_min_delta = 1e-3

trainable_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

optimizer = AdamW(trainable_parameters, lr=learning_rate, weight_decay=weight_decay)

optimizer_steps_per_epoch = math.ceil(len(train_loader) / gradient_accumulation_steps)
total_optimizer_steps = optimizer_steps_per_epoch * max_epochs
warmup_steps = int(total_optimizer_steps * warmup_ratio)

print("Optimizer steps per epoch:", optimizer_steps_per_epoch)
print("Total optimizer steps:", total_optimizer_steps)
print("Warmup steps:", warmup_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps,
)

trainer = ManualQLoRATrainer(
    model=model,
    optimizer=optimizer,
    device=DEVICE,
    train_loader=train_loader,
    validation_loader=validation_loader,
    gradient_accumulation_steps=gradient_accumulation_steps,
    max_grad_norm=1.0,
    scheduler=scheduler,
)

Optimizer steps per epoch: 150
Total optimizer steps: 1200
Warmup steps: 60


In [48]:
# One epoch valid try after restart kernel
torch.cuda.reset_peak_memory_stats()

train_metric = trainer.train_epoch()
val_metric = trainer.evaluate()

print(train_metric)
print(val_metric)

peak_memory_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print(f"Peak VRAM: {peak_memory_gb:.2f} GB")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'train_loss': 0.3822341536102096}
{'validation_loss': 0.3268331232421228}
Peak VRAM: 9.06 GB


In [26]:
# One epoch valid try after restart kernel + accumuleted gradient
import math

train_metrics = trainer.train_epoch()
validation_metrics = trainer.evaluate()

expected_steps = math.ceil(
    len(train_loader)
    / gradient_accumulation_steps
)

print(train_metrics)
print(validation_metrics)
print("Expected optimizer steps:", expected_steps)

assert train_metrics["optimizer_steps"] == expected_steps
assert expected_steps == 150

{'train_loss': 0.48091040773032373, 'mean_gradient_norm': 1.886342950661977, 'learning_rate': 9.210526315789474e-05, 'optimizer_steps': 150, 'global_step': 150}
{'validation_loss': 0.345728165995982}
Expected optimizer steps: 150


In [24]:
torch.cuda.reset_peak_memory_stats()

history = trainer.fit(
    num_epochs=8,
    output_dir=str(OUTPUTS_DIR / "qlora_manual_fp32_token_aware_v1"),
    early_stopping_patience=2,
    early_stopping_min_delta=1e-3,
)

print(history)

peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3
print(f"Peak VRAM: {peak_vram_gb:.2f} GB")

Train:   0%|          | 0/300 [00:00<?, ?microbatch/s]

Validation:   0%|          | 0/38 [00:00<?, ?batch/s]

Epoch 1/8 | train_loss=0.4809 | validation_loss=0.3446 | grad_norm=1.8577 | lr=9.21e-05
LoRA adapter сохранён в: /content/drive/MyDrive/LLMProjects/fine-tune/outputs/qlora_manual_fp32_token_aware_v1/best_adapter


Train:   0%|          | 0/300 [00:00<?, ?microbatch/s]

Validation:   0%|          | 0/38 [00:00<?, ?batch/s]

...

Validation:   0%|          | 0/38 [00:00<?, ?batch/s]

Epoch 5/8 | train_loss=0.1789 | validation_loss=0.3094 | grad_norm=1.7081 | lr=3.95e-05
Validation loss не улучшился. Bad epochs: 1


Train:   0%|          | 0/300 [00:00<?, ?microbatch/s]

Validation:   0%|          | 0/38 [00:00<?, ?batch/s]

Epoch 6/8 | train_loss=0.1482 | validation_loss=0.3301 | grad_norm=1.8058 | lr=2.63e-05
Validation loss не улучшился. Bad epochs: 2
Early stopping: validation loss не улучшался 2 эпохи подряд.
В память восстановлен adapter с лучшим validation loss.
[{'epoch': 1, 'train_loss': 0.4808929137153047, 'mean_gradient_norm': 1.8577321982383728, 'learning_rate': 9.210526315789474e-05, 'optimizer_steps': 150, 'global_step': 150, 'validation_loss': 0.34460029185881175}, {'epoch': 2, 'train_loss': 0.29664219518227164, 'mean_gradient_norm': 1.4527753281593323, 'learning_rate': 7.894736842105263e-05, 'optimizer_steps': 150, 'global_step': 300, 'validation_loss': 0.3114177066639163}, {'epoch': 3, 'train_loss': 0.249290832505776, 'mean_gradient_norm': 1.4572034454345704, 'learning_rate': 6.578947368421054e-05, 'optimizer_steps': 150, 'global_step': 450, 'validation_loss': 0.3023523535080353}, {'epoch': 4, 'train_loss': 0.2115725834205986, 'mean_gradient_norm': 1.5698962132136027, 'learning_rate': 5.26

In [25]:
RUN_DIR = OUTPUTS_DIR / "qlora_manual_fp32_token_aware_v1"
BEST_ADAPTER_DIR = RUN_DIR / "best_adapter"

RUN_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_adapters(BEST_ADAPTER_DIR)

tokenizer.save_pretrained(BEST_ADAPTER_DIR)

LoRA adapter сохранён в: /content/drive/MyDrive/LLMProjects/fine-tune/outputs/qlora_manual_fp32_token_aware_v1/best_adapter


('/content/drive/MyDrive/LLMProjects/fine-tune/outputs/qlora_manual_fp32_token_aware_v1/best_adapter/tokenizer_config.json',
 '/content/drive/MyDrive/LLMProjects/fine-tune/outputs/qlora_manual_fp32_token_aware_v1/best_adapter/chat_template.jinja',
 '/content/drive/MyDrive/LLMProjects/fine-tune/outputs/qlora_manual_fp32_token_aware_v1/best_adapter/tokenizer.json')

In [26]:
history_path = RUN_DIR / "history.json"

with history_path.open("w", encoding="utf-8") as file:
    json.dump(
        history,
        file,
        ensure_ascii=False,
        indent=2,
    )

In [27]:
best_epoch_metrics = min(
    history,
    key=lambda metrics: metrics["validation_loss"],
)

summary = {
    "best_epoch": best_epoch_metrics["epoch"],
    "best_validation_loss": best_epoch_metrics["validation_loss"],
    "best_train_loss": best_epoch_metrics["train_loss"],
    "best_global_step": best_epoch_metrics["global_step"],
    "num_completed_epochs": len(history),
}

with (RUN_DIR / "summary.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(summary)

{'best_epoch': 4, 'best_validation_loss': 0.29999137369212064, 'best_train_loss': 0.2115725834205986, 'best_global_step': 600, 'num_completed_epochs': 6}


In [29]:
run_config = {
    "model_name": MODEL_NAME,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "micro_batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "effective_batch_size": (
        BATCH_SIZE * gradient_accumulation_steps
    ),
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "max_grad_norm": 1.0,
    "max_epochs": max_epochs,
    "warmup_ratio": warmup_ratio,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "bnb_compute_dtype": str(COMPUTE_DTYPE),
    "model_dtype": str(COMPUTE_DTYPE),
    "use_amp": trainer.use_amp,
    "amp_dtype": str(trainer.dtype),
    "gradient_checkpointing": False,
    "warmup_steps": warmup_steps,
    "total_optimizer_steps": total_optimizer_steps,
    "early_stopping_patience": early_stopping_patience,
    "early_stopping_min_delta": early_stopping_min_delta,
}

with (RUN_DIR / "config.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        run_config,
        file,
        ensure_ascii=False,
        indent=2,
    )

# Inference Qwen-QLoRA


In [6]:
import torch

import json
from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

PROJECT_ROOT_LOCAL_MACHINE = Path.cwd().parent

RUN_DIR = PROJECT_ROOT_LOCAL_MACHINE / "qlora_manual_outputs"
BEST_ADAPTER_DIR = RUN_DIR / "best_adapter"

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    dtype=COMPUTE_DTYPE,
)

inference_model = PeftModel.from_pretrained(
    base_model,
    BEST_ADAPTER_DIR,
    is_trainable=False,
)

inference_model.config.use_cache = True
inference_model.eval()

inference_tokenizer = AutoTokenizer.from_pretrained(BEST_ADAPTER_DIR)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

inference_tokenizer.padding_side = "right"

MODEL_DEVICE = next(inference_model.parameters()).device

print("Model device:", MODEL_DEVICE)
print("Model class:", inference_model.__class__.__name__)
print("Tokenizer:", inference_tokenizer.__class__.__name__)
print("Pad token:", inference_tokenizer.pad_token)
print("Eos token:", inference_tokenizer.eos_token)

C:\Users\Артём\PycharmProjects\routing-support-tickets\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights:   0%|          | 1/290 [00:00<01:12,  3.98it/s]C:\Users\Артём\PycharmProjects\routing-support-tickets\.venv\Lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
C:\Users\Артём\PycharmProjects\routing-support-tickets\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 290/290 [00:33<00:00,  8.79it/s]


Model device: cpu
Model class: PeftModelForCausalLM
Tokenizer: Qwen2Tokenizer
Pad token: <|endoftext|>
Eos token: <|im_end|>


In [7]:
TEST_PATH = PROJECT_ROOT_LOCAL_MACHINE / "data" / "processed" / "test.jsonl"

assert TEST_PATH.exists(), f"Не найден test: {TEST_PATH}"

test_dataset = load_dataset(
    "json",
    data_files=str(TEST_PATH),
    split="train",
)

print("Test examples:", len(test_dataset))
print("Columns:", test_dataset.column_names)

Generating train split: 150 examples [00:00, 7925.05 examples/s]

Test examples: 150
Columns: ['messages', 'target']


In [8]:
GENERATION_CONFIG = {
    "max_new_tokens": 256,
    "do_sample": False,
    "use_cache": True,
    "eos_token_id": inference_tokenizer.eos_token_id,
    "pad_token_id": inference_tokenizer.pad_token_id,
}

MODEL_DEVICE = next(inference_model.parameters()).device

def generate_ticket_routing(messages: list[dict[str, str]]) -> str:
    assert len(messages) >= 2
    assert [message["role"] for message in messages[:2]] == ["system", "user"]

    prompt_messages = messages[:2]

    model_inputs = inference_tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )

    model_inputs = {
        name: tensor.to(MODEL_DEVICE)
        for name, tensor in model_inputs.items()
    }

    prompt_length = model_inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generate_ids = inference_model.generate(
            **model_inputs,
            **GENERATION_CONFIG,
        )

    new_token_ids = generate_ids[0, prompt_length:]

    return inference_tokenizer.decode(
        new_token_ids,
        skip_special_tokens=True,
    ).strip()

In [9]:
test_example = test_dataset[0]

print("Input")
print(test_example["messages"][1]["content"][:1000])

raw_prediction = generate_ticket_routing(test_example["messages"])

print("Raw model output")
print(raw_prediction)

try:
    parsed = json.loads(raw_prediction)

    print("\nParsed JSON")
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
except json.JSONDecodeError as error:
    print("\nJSON parse error:",error)

Input
Subject: Payment Details for Billing

Body: Dear Customer Support,

I am reaching out to request a detailed statement of billing payments for my recent acquisitions of hardware and software, including peripheral devices and computers. These items are intended for healthcare procurement, and I require a complete invoice with itemized details to ensure proper accounting and compliance with internal procedures. Furthermore, please include the applicable payment guidelines and relevant terms and conditions related to these purchases. Your prompt assistance would be highly appreciated to facilitate efficient processing.

Thank you.


C:\Users\Артём\PycharmProjects\routing-support-tickets\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
C:\Users\Артём\PycharmProjects\routing-support-tickets\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Raw model output
{"ticket_type": "Request", "topic": "Billing and Payments", "urgency": "medium", "tags": ["Billing", "Payment", "Hardware", "Software", "Account"]}

Parsed JSON
{
  "ticket_type": "Request",
  "topic": "Billing and Payments",
  "urgency": "medium",
  "tags": [
    "Billing",
    "Payment",
    "Hardware",
    "Software",
    "Account"
  ]
}


In [10]:
from src.prediction_utils import parse_and_validate_response

PREDICTIONS_DIR = RUN_DIR / "predictions"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = PREDICTIONS_DIR / "qlora_manual_token_aware_v1_test.jsonl"

with PREDICTIONS_PATH.open("w", encoding="utf-8") as file:
    progress = tqdm(
        test_dataset,
        total=len(test_dataset),
        desc="Generating test predictions",
        unit="ticket",
    )

    for example in progress:
        raw_prediction = generate_ticket_routing(example["messages"])

        prediction_record = parse_and_validate_response(raw_prediction)

        file.write(json.dumps(prediction_record , ensure_ascii=False) + "\n")

print(f"Predictions saved to: {PREDICTIONS_PATH}")

Generating test predictions: 100%|██████████| 150/150 [1:57:46<00:00, 47.11s/ticket] 

Predictions saved to: C:\Users\Артём\PycharmProjects\routing-support-tickets\qlora_manual_outputs\predictions\qlora_manual_token_aware_v1_test.jsonl


In [11]:
with PREDICTIONS_PATH.open("r", encoding="utf-8") as file:
    for _ in range(3):
        print(file.readline().rstrip())

{"raw_response": "{\"ticket_type\": \"Request\", \"topic\": \"Billing and Payments\", \"urgency\": \"medium\", \"tags\": [\"Billing\", \"Payment\", \"Hardware\", \"Software\", \"Account\"]}", "parsed_response": {"ticket_type": "Request", "topic": "Billing and Payments", "urgency": "medium", "tags": ["Billing", "Payment", "Hardware", "Software", "Account"]}, "json_valid": true, "schema_valid": true, "prediction": {"ticket_type": "Request", "topic": "Billing and Payments", "urgency": "medium", "tags": ["Billing", "Payment", "Hardware", "Software", "Account"]}, "error": null}
{"raw_response": "{\"ticket_type\": \"Change\", \"topic\": \"Product Support\", \"urgency\": \"medium\", \"tags\": [\"Feedback\", \"Marketing\", \"Strategy\", \"Campaign\", \"Brand Development\"]}", "parsed_response": {"ticket_type": "Change", "topic": "Product Support", "urgency": "medium", "tags": ["Feedback", "Marketing", "Strategy", "Campaign", "Brand Development"]}, "json_valid": true, "schema_valid": true, "pre